In [ ]:
!npm install -g localtunnel


In [ ]:
import os
os.chdir('/content')
!rm -rf /content/venv
!pip install virtualenv
!virtualenv /content/venv
!/content/venv/bin/pip install fastapi uvicorn requests pydantic
!rm -rf /content/usman-and-umar-app
!git clone --single-branch --branch feature/hybrid-task-orchestrator https://github.com/saedhawwa23-rgb/usman-and-umar-app.git /content/usman-and-umar-app
print("Fast backend ready.")


In [ ]:
import os, time
if os.path.exists('/content/usman-and-umar-app/cloud-backend'):
    os.chdir('/content/usman-and-umar-app/cloud-backend')
else:
    raise Exception("Backend folder missing. Cell 2 failed to clone the repo.")
!/content/venv/bin/python -m uvicorn main:app --host 0.0.0.0 --port 7860 > /tmp/fastapi.log 2>&1 &
time.sleep(5)
print("FastAPI Server is running.")


### Configure a per-server Localtunnel subdomain

In Google Colab, open **Secrets** (key icon), add a secret named `TUNNEL_SUBDOMAIN`, enter a unique value such as `usman-umar-1` for Server 1 or `usman-umar-2` for Server 2, and enable notebook access. If unset, the tunnel cell defaults to `usman-umar-1`.


In [ ]:
import os, time
try:
    from google.colab import userdata
    os.environ["TUNNEL_SUBDOMAIN"] = userdata.get("TUNNEL_SUBDOMAIN")
except Exception:
    pass  # Allows the environment-variable/default fallback outside Colab or when the secret is unavailable.
subdomain = os.environ.get("TUNNEL_SUBDOMAIN", "usman-umar-1")
!lt --port 7860 --subdomain {subdomain} > /tmp/localtunnel.log 2>&1 &
time.sleep(3)
print(f"Public URL: https://{subdomain}.loca.lt")
print("Open the URL in Chrome first to bypass the Localtunnel warning page.")


In [ ]:
# Includes langchain-openai, required by OpenSwarm's ChatOpenAI model adapter.
!/content/venv/bin/pip install -r /content/usman-and-umar-app/cloud-backend/requirements-ai.txt > /tmp/pip_ai.log 2>&1 &
print("Downloading OpenSwarm AI packages in the background. You can use the app now; AI tasks will work once the download finishes.")


In [ ]:
import importlib.util, subprocess, sys, time
ready = False
for _ in range(6):
    check = subprocess.run(['/content/venv/bin/python', '-c', "import importlib.util,sys; sys.exit(0 if importlib.util.find_spec('langchain') else 1)"], check=False)
    if check.returncode == 0:
        print('✅ OpenSwarm AI Engine is ready!')
        ready = True
        break
    time.sleep(5)
if not ready:
    print('⏳ AI Engine is still downloading...')
